In [2]:
# calculate the whole ecosystem, tree, shrub, sphagnum GPP
# calculate the GPPmax and CUP
# daily gpp
# date, year, doy, variable (GPP) as model name

import pandas as pd

path_acc = "../../data_results/1_data_source/1_daily_simulations/"
ls_plots = ["P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"]
# ls_plots = ["P08"]

# get the date from 2015-01-01 to 2021-12-31
date_range = pd.date_range(start='2011-01-01', end='2021-12-31', freq='D')
df = pd.DataFrame(date_range, columns=['date'])
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day']   = df['date'].dt.day
df          = df[~((df['month'] == 2) & (df['day'] == 29))]  # delete the 2/29
df['date']  = pd.to_datetime(df['date'])

def summary_gpp(isps=None):
    test_pd = pd.DataFrame(columns=["date", "year", "doy", "P04", "P06", "P08", "P10", "P11", "P13", "P16", "P17", "P19", "P20"])
    test_pd["date"] = df['date']
    test_pd["year"] = df['date'].dt.year
    test_pd["doy"]  = df['date'].dt.dayofyear
    # read data
    for idx_plot, iplot in enumerate(ls_plots):
        pd_data = pd.read_csv(path_acc + "TECO-SPRUCE_run_mcmc_alltreat_"+iplot+"_Daily.csv")
        pd_data['datetime'] = pd.to_datetime(pd_data[' year'].astype(str) + '-01-01') + pd.to_timedelta(pd_data['doy'] - 1, unit='D')
        if isps == "ecosystem":
            pd_data["GPP"] = (0.25*pd_data["gpp_Shrub"] + 0.25*pd_data["gpp_Sphagnum"] + 0.5*pd_data["gpp_Tree"])*24
        elif isps == "tree":
            pd_data["GPP"] = (0.5*pd_data["gpp_Tree"])*24
        elif isps == "shrub":
            pd_data["GPP"] = (0.25*pd_data["gpp_Shrub"])*24
        elif isps == "sphagnum":
            pd_data["GPP"] = (0.25*pd_data["gpp_Sphagnum"])*24
        # 
        pd_data = pd_data[["datetime", "GPP"]]
        pd_data.set_index('datetime', inplace=True)
        pd_data = pd_data.loc['2011':'2021']
        daily_mean = pd_data.resample('D').mean()
        daily_mean = daily_mean.reset_index()
        test_pd[iplot] = daily_mean["GPP"]
    # save data
    test_pd.to_excel(f"../../data_results/2_results/2-1_simu_gpp_time_series/TECO-SPRUCE_DA_{isps}_2011-2021.xlsx") 

for idx_sps, isps in enumerate(["ecosystem", "tree", "shrub", "sphagnum"]):
    print(isps)
    summary_gpp(isps)


ecosystem
tree
shrub
sphagnum
